# Tabular XGBoost Hurdle Model Experimentation
**Objective:** Engineer tabular metadata, build a robust scikit-learn preprocessing pipeline, and implement a Two-Stage Classification-Regression Cascade (Hurdle Model) to solve zero-inflated retail demand.

**Architecture Overview:**
* **Stage 1 (Gatekeeper):** XGBClassifier predicts if an item will be a Viral Hit (>= 50 units) or Baseline (< 50 units).
* **Stage 2A (Baseline Regressor):** XGBRegressor trained only on non-viral items.
* **Stage 2B (Viral Regressor):** XGBRegressor trained only on viral items using Log1p transformation.

**Professional Standards Implemented:**
* Complete pipeline integration (Imputation + Target Encoding) to prevent data leakage.
* Stratified train-test splitting and lifecycle-bounded time-series skeletons.
* Precision-Recall threshold tuning for highly imbalanced classification.

## Imports and Setup

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import category_encoders as ce

import warnings
warnings.filterwarnings('ignore')

## Data Loading and Merging

### Loading Data

In [3]:
targets_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/processed/time_series_targets.csv'),
                        dtype= {'article_id': str})

cnn_scores_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/processed/cnn_visual_scores.csv'),
                        dtype= {'article_id': str})

articles_df= pd.read_csv(filepath_or_buffer= os.path.abspath('../data/raw/articles.csv'),
                        dtype= {'article_id': str})

In [4]:
targets_df.head()

,article_id,month,monthly_sales
0,0108775015,2018-09,662
1,0108775015,2018-10,1532
2,0108775015,2018-11,1660
3,0108775015,2018-12,1207
4,0108775015,2019-01,1522


In [5]:
targets_df.shape

(768883, 3)

In [6]:
cnn_scores_df.head()

,article_id,CNN_Visual_Trend_Score
0,0111593001,0.305146
1,0112679048,0.236609
2,0114428030,0.389910
3,0118458004,0.275760
4,0118458028,0.230257


In [7]:
cnn_scores_df.shape

(9961, 2)

In [8]:
articles_df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [9]:
articles_df.shape

(105542, 25)

In [10]:
print(pd.to_datetime(targets_df['month']).min())
print(pd.to_datetime(targets_df['month']).max())
print(pd.to_datetime(targets_df['month']).max() - pd.to_datetime(targets_df['month']).min())

2018-09-01 00:00:00
2020-09-01 00:00:00
731 days 00:00:00


In [11]:
9961 * (np.ceil(731/ 30))

np.float64(249025.0)

* Note:
- As we can see above, unique articles are 9961 (form cnn_scores_df.shape).
- If all items were available for all months (Starting from 2018-09 to 2020-09), there should be approximately (9961 * 25) rows in targets_df, which is not the case.
- Some item could be introduced after the min (2018-09) or were taken off shelf before max (2020-09) or both.
- We should find min and max months for each itesm, i.e. The month the item was introduced and the month it was taken off shelf (if it was), and fill all the month between those with 0.

### Creating Skeleton df with sales 0 for months missing foe each Article (between the months of article introduction and item was taken off shelf)

In [12]:
# Getting First and Last Month of Sales for Each Article:
lifecycle= targets_df.groupby('article_id')['month'].agg(['min', 'max']).reset_index()

# Getting Sorted list of all Possible Months in Dataset:
all_months= sorted(targets_df['month'].unique())

# Creating a Brute-Force Skeleton DF with all Months for all Items:
skeleton= pd.MultiIndex.from_product(
    [targets_df['article_id'].unique(), all_months],
    names= ['article_id', 'month']
).to_frame(index= False)

# Attaching Life-Cycle Bounds to Skeleton DF:
skeleton= pd.merge(left= skeleton,
                   right= lifecycle,
                   on= 'article_id',
                   how= 'inner')

# Filtering only the rows where the month falls within the item's active lifecycle:
bounded_skeleton= skeleton[
    (skeleton['month'] >= skeleton['min']) &
    (skeleton['month'] <= skeleton['max'])
][['article_id', 'month']]

# Merging Actual Monthly Sales onto Bounded Skeleton DF:
targets_df_fixed= pd.merge(left= bounded_skeleton,
                           right= targets_df,
                           on= ['article_id', 'month'],
                           how= 'left')

# Filling Missing Months Sales for Each Item with 0:
targets_df_fixed['monthly_sales'] = targets_df_fixed['monthly_sales'].fillna(0)

targets_df_fixed.shape

(933800, 3)

### Merging Data

#### First Merge

In [13]:
df= pd.merge(left= targets_df_fixed,
             right= cnn_scores_df,
             on= 'article_id',
             how= 'inner')

In [14]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score
0,0111593001,2018-09,530.0,0.305146
1,0111593001,2018-10,1207.0,0.305146
2,0111593001,2018-11,1027.0,0.305146
3,0111593001,2018-12,1558.0,0.305146
4,0111593001,2019-01,504.0,0.305146


In [15]:
df.shape

(88760, 4)

#### Final Merge

In [16]:
articles_df.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

In [17]:
articles_df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [18]:
columns_to_merge= [
    'article_id', 'prod_name', 'product_type_name', 'product_group_name',
    'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name',
    'perceived_colour_master_name', 'department_name', 'index_name',
    'index_group_name', 'section_name', 'garment_group_name'
]

In [19]:
# Final Merge:
df = pd.merge(left= df,
              right= articles_df[columns_to_merge],
              on= 'article_id',
              how= 'inner')

In [20]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name
0,0111593001,2018-09,530.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
1,0111593001,2018-10,1207.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
2,0111593001,2018-11,1027.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
3,0111593001,2018-12,1558.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights
4,0111593001,2019-01,504.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights


In [21]:
df.shape

(88760, 16)

In [22]:
# Adding Month as a Separate Column:
df['month_int'] = pd.to_datetime(df['month']).dt.month

In [23]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_int
0,0111593001,2018-09,530.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,9
1,0111593001,2018-10,1207.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,10
2,0111593001,2018-11,1027.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,11
3,0111593001,2018-12,1558.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,12
4,0111593001,2019-01,504.0,0.305146,Support 40 den 1p Tights,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,1


In [24]:
df.shape

(88760, 17)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88760 entries, 0 to 88759
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   article_id                    88760 non-null  object 
 1   month                         88760 non-null  object 
 2   monthly_sales                 88760 non-null  float64
 3   CNN_Visual_Trend_Score        88760 non-null  float64
 4   prod_name                     88760 non-null  object 
 5   product_type_name             88760 non-null  object 
 6   product_group_name            88760 non-null  object 
 7   graphical_appearance_name     88760 non-null  object 
 8   colour_group_name             88760 non-null  object 
 9   perceived_colour_value_name   88760 non-null  object 
 10  perceived_colour_master_name  88760 non-null  object 
 11  department_name               88760 non-null  object 
 12  index_name                    88760 non-null  object 
 13  i

In [26]:
# Checking Uniques in all Object Type Columns to address possible Dimensionality Issue:
for col in df.select_dtypes(include= 'object'):
    print(f"{col} - Uniques: {df[col].nunique()}\n")

article_id - Uniques: 9961

month - Uniques: 25

prod_name - Uniques: 8457

product_type_name - Uniques: 99

product_group_name - Uniques: 15

graphical_appearance_name - Uniques: 30

colour_group_name - Uniques: 50

perceived_colour_value_name - Uniques: 8

perceived_colour_master_name - Uniques: 19

department_name - Uniques: 235

index_name - Uniques: 10

index_group_name - Uniques: 5

section_name - Uniques: 55

garment_group_name - Uniques: 21



In [27]:
# Dropping 'prod_name' column for having too many categories, working essentially as unique:
df= df.drop('prod_name', axis= 1)

In [28]:
df.head()

,article_id,month,monthly_sales,CNN_Visual_Trend_Score,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_int
0,0111593001,2018-09,530.0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,9
1,0111593001,2018-10,1207.0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,10
2,0111593001,2018-11,1027.0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,11
3,0111593001,2018-12,1558.0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,12
4,0111593001,2019-01,504.0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,1


In [29]:
# Checking Uniques in all Object Type Columns to address possible Dimensionality Issue:
for col in df.select_dtypes(include= 'object'):
    print(f"{col} - Uniques: {df[col].nunique()}\n")

article_id - Uniques: 9961

month - Uniques: 25

product_type_name - Uniques: 99

product_group_name - Uniques: 15

graphical_appearance_name - Uniques: 30

colour_group_name - Uniques: 50

perceived_colour_value_name - Uniques: 8

perceived_colour_master_name - Uniques: 19

department_name - Uniques: 235

index_name - Uniques: 10

index_group_name - Uniques: 5

section_name - Uniques: 55

garment_group_name - Uniques: 21



## Data Pre-Processing Pipeline

### Features and Target

In [30]:
# Defining Target and Features:
VIRAL_THRESHOLD = 50
df['is_viral'] = (df['monthly_sales'] >= VIRAL_THRESHOLD).astype(int)

X = df.drop(['article_id', 'month', 'monthly_sales', 'is_viral'], axis=1)
y_class = df['is_viral']
y_reg = df['monthly_sales'] # Kept to route data later


In [31]:
X.head()

,CNN_Visual_Trend_Score,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,month_int
0,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,9
1,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,10
2,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,11
3,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,12
4,0.305146,Underwear Tights,Socks & Tights,Solid,Black,Dark,Black,Tights basic,Lingeries/Tights,Ladieswear,"Womens Nightwear, Socks & Tigh",Socks and Tights,1


In [32]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88760 entries, 0 to 88759
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   CNN_Visual_Trend_Score        88760 non-null  float64
 1   product_type_name             88760 non-null  object 
 2   product_group_name            88760 non-null  object 
 3   graphical_appearance_name     88760 non-null  object 
 4   colour_group_name             88760 non-null  object 
 5   perceived_colour_value_name   88760 non-null  object 
 6   perceived_colour_master_name  88760 non-null  object 
 7   department_name               88760 non-null  object 
 8   index_name                    88760 non-null  object 
 9   index_group_name              88760 non-null  object 
 10  section_name                  88760 non-null  object 
 11  garment_group_name            88760 non-null  object 
 12  month_int                     88760 non-null  int32  
dtypes

### Defining Different Features for Preprocessing:


In [33]:
## Numeric Features:
num_features= ['CNN_Visual_Trend_Score', 'month_int']

## High Cardinality (Dimensionality) Categorical Features (> 20 uniques):
high_cat_features= [
    'product_type_name', 'graphical_appearance_name', 
    'colour_group_name', 'department_name', 
    'section_name', 'garment_group_name'
]

## Low Cardinality (Dimensionality) Categorical Features: (<= 20 Uniques):
low_cat_features= [
    'product_group_name', 'perceived_colour_value_name',
    'perceived_colour_master_name', 'index_name', 'index_group_name'
]


### Preprocessing Pipeline for Different Features

In [34]:
# Numeric Features:
num_transformer= Pipeline(steps= [
    ('imputer', SimpleImputer(strategy= 'median'))
])

# High Cardinality Categorical Features:
high_cat_transformer= Pipeline(steps= [
    ('imputer', SimpleImputer(strategy= 'constant', fill_value= 'Unknown')),
    ('target_enc', ce.TargetEncoder(smoothing= 10))
])

# Low Cardinality Categorical Features:
low_cat_transformer= Pipeline(steps= [
    ('imputer', SimpleImputer(strategy= 'constant', fill_value= 'Unknown')),
    ('onehot', OneHotEncoder(handle_unknown= 'ignore', sparse_output= False))
])

### Final Data Pre-Processor

In [35]:
preprocessor= ColumnTransformer(
    transformers= [
        ('num', num_transformer, num_features),
        ('low_cat', low_cat_transformer, low_cat_features),
        ('high_cat', high_cat_transformer, high_cat_features)
    ]
)

## Stage 1: The Gatekeeper (XGBClassifier)

In [36]:
# Final Pipeline:
pipeline= Pipeline(steps= [
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(
        objective= 'binary:logistic',
        tree_method= 'hist',
        device= 'cuda',
        random_sate= 42,
        eval_metric= 'auc'
    ))
])

### RandomizedSearchCV on XGBoost (Gate-Keeper Model)

#### Train-Test Split

In [37]:
X_train, X_val, y_train_class, y_val_class, y_train_reg, y_val_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)


In [38]:
print(X_train.shape)
print(X_val.shape)
print(y_train_class.shape)
print(y_val_class.shape)
print(y_train_reg.shape)
print(y_val_reg.shape)

(71008, 13)
(17752, 13)
(71008,)
(17752,)
(71008,)
(17752,)


#### Sampling 20% of Training Data for RandomizedSearch to Save Time

In [39]:
X_search, _, y_search_class, _ = train_test_split(
    X_train, y_train_class, train_size= 0.3, random_state= 42, stratify= y_train_class
)

In [40]:
print(X_search.shape)
print(y_search_class.shape)


(21302, 13)
(21302,)


#### Randomized Search

In [41]:
# Parameter Grid:
scale_weight_estimate = len(y_train_class[y_train_class == 0]) / len(y_train_class[y_train_class == 1])

param_grid = {
    'model__max_depth': [4, 6, 8, 10],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__n_estimators': [100, 300, 500],
    'model__scale_pos_weight': [1, scale_weight_estimate, scale_weight_estimate * 1.5],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

In [42]:
# 3-Fold Cross Validation:
random_search= RandomizedSearchCV(estimator= pipeline,
                          param_distributions= param_grid,
                          n_iter= 100,
                          cv= 3,
                          scoring= 'roc_auc',
                          verbose= 2,
                          n_jobs= -1,
                          random_state= 42)

In [43]:
# Model Training:
import time

print('Starting Hyper-Parameter Tuning..')
start_time= time.perf_counter()

random_search.fit(X_search, y_search_class)

end_time= time.perf_counter()
total_training_duration= end_time - start_time

print(f'Tuning finished in: {total_training_duration/60:.2f} minutes!')
print(f"Best Classifier Params: {random_search.best_params_}")

Starting Hyper-Parameter Tuning..
Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.1, model__max_depth=4, model__n_estimators=100, model__scale_pos_weight=8.427486252213626, model__subsample=0.8; total time=   8.4s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=4, model__n_estimators=100, model__scale_pos_weight=5.618324168142418, model__subsample=1.0; total time=  10.0s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.1, model__max_depth=4, model__n_estimators=100, model__scale_pos_weight=8.427486252213626, model__subsample=0.8; total time=   9.9s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=4, model__n_estimators=100, model__scale_pos_weight=5.618324168142418, model__subsample=1.0; total time=  10.3s
[CV] END model__colsample_bytree=0.8, model__learning_rate=0.1, model__max_depth=4, model__n_estimators=100, model__scale_pos_wei

#### Model Evaluation

In [44]:
# Evaluate Optimal Stage 1 Classifier
best_clf = random_search.best_estimator_
print("Training optimal Stage 1 Gatekeeper on full training data...")
best_clf.fit(X_train, y_train_class)

preds_class = best_clf.predict(X_val)
preds_proba = best_clf.predict_proba(X_val)[:, 1]

print("\n--- Stage 1: Gatekeeper Performance ---")
print(f"ROC-AUC Score: {roc_auc_score(y_val_class, preds_proba):.3f}\n")
print(classification_report(y_val_class, preds_class, target_names=['Baseline (<50)', 'Viral Hit (>=50)']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_class, preds_class))

Training optimal Stage 1 Gatekeeper on full training data...

--- Stage 1: Gatekeeper Performance ---
ROC-AUC Score: 0.871

                  precision    recall  f1-score   support

  Baseline (<50)       0.88      0.97      0.93     15070
Viral Hit (>=50)       0.65      0.27      0.38      2682

        accuracy                           0.87     17752
       macro avg       0.76      0.62      0.65     17752
    weighted avg       0.85      0.87      0.84     17752


Confusion Matrix:
[[14670   400]
 [ 1952   730]]


In [45]:
# 1. Get probabilities instead of hard 0/1 predictions
y_probs = best_clf.predict_proba(X_val)[:, 1]

# 2. Test a lower threshold (e.g., 0.3 instead of 0.5)
NEW_THRESHOLD = 0.3
preds_adjusted = (y_probs >= NEW_THRESHOLD).astype(int)

print(f"\n--- Performance at Threshold {NEW_THRESHOLD} ---")
print(classification_report(y_val_class, preds_adjusted, target_names=['Baseline (<50)', 'Viral Hit (>=50)']))


--- Performance at Threshold 0.3 ---
                  precision    recall  f1-score   support

  Baseline (<50)       0.93      0.88      0.91     15070
Viral Hit (>=50)       0.49      0.61      0.54      2682

        accuracy                           0.84     17752
       macro avg       0.71      0.75      0.72     17752
    weighted avg       0.86      0.84      0.85     17752



**Keeping the Threshold at 0.3**